# Q-Learning Algorithm

Implementation of Q-Learning, a reinforcement learning algorithm used to find the optimal action-selection policy for any given finite Markov decision process (MDP).

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import random

## Q-Learning Agent Class

In [ ]:
class QLearningAgent:
    """Q-Learning Agent for Reinforcement Learning"""
    
    def __init__(self, learning_rate=0.1, discount_factor=0.99, epsilon=0.1):
        """
        Initialize the Q-Learning Agent
        
        Args:
            learning_rate (float): Learning rate (alpha)
            discount_factor (float): Discount factor (gamma)
            epsilon (float): Exploration rate
        """
        self.learning_rate = learning_rate
        self.discount_factor = discount_factor
        self.epsilon = epsilon
        self.q_table = defaultdict(lambda: defaultdict(float))
    
    def get_action(self, state, actions):
        """
        Epsilon-greedy action selection
        
        Args:
            state: Current state
            actions: Available actions
            
        Returns:
            Selected action
        """
        if random.random() < self.epsilon:
            return random.choice(actions)
        else:
            q_values = [self.q_table[state][action] for action in actions]
            max_q = max(q_values)
            return random.choice([actions[i] for i, q in enumerate(q_values) if q == max_q])
    
    def update_q_value(self, state, action, reward, next_state, next_actions):
        """
        Update Q-value using the Q-Learning formula:
        Q(s,a) = Q(s,a) + alpha * (reward + gamma * max(Q(s',a')) - Q(s,a))
        
        Args:
            state: Current state
            action: Action taken
            reward: Reward received
            next_state: Next state
            next_actions: Available actions in next state
        """
        current_q = self.q_table[state][action]
        max_next_q = max([self.q_table[next_state][a] for a in next_actions]) if next_actions else 0
        
        new_q = current_q + self.learning_rate * (reward + self.discount_factor * max_next_q - current_q)
        self.q_table[state][action] = new_q

## Example: Simple Grid World

In [ ]:
class GridWorld:
    """Simple Grid World Environment"""
    
    def __init__(self, size=5, goal=(4, 4)):
        self.size = size
        self.goal = goal
        self.agent_pos = (0, 0)
    
    def reset(self):
        self.agent_pos = (0, 0)
        return self.agent_pos
    
    def step(self, action):
        """
        Actions: 0=up, 1=down, 2=left, 3=right
        """
        x, y = self.agent_pos
        
        if action == 0:  # up
            x = max(0, x - 1)
        elif action == 1:  # down
            x = min(self.size - 1, x + 1)
        elif action == 2:  # left
            y = max(0, y - 1)
        elif action == 3:  # right
            y = min(self.size - 1, y + 1)
        
        self.agent_pos = (x, y)
        
        # Reward: +1 for reaching goal, -0.01 for each step
        reward = 1.0 if self.agent_pos == self.goal else -0.01
        done = self.agent_pos == self.goal
        
        return self.agent_pos, reward, done
    
    def get_actions(self):
        return [0, 1, 2, 3]

## Training Loop

In [ ]:
# Initialize environment and agent
env = GridWorld(size=5, goal=(4, 4))
agent = QLearningAgent(learning_rate=0.1, discount_factor=0.99, epsilon=0.1)

# Training parameters
episodes = 100
max_steps = 100
rewards_per_episode = []

# Training loop
for episode in range(episodes):
    state = env.reset()
    episode_reward = 0
    
    for step in range(max_steps):
        action = agent.get_action(state, env.get_actions())
        next_state, reward, done = env.step(action)
        
        agent.update_q_value(state, action, reward, next_state, env.get_actions())
        
        episode_reward += reward
        state = next_state
        
        if done:
            break
    
    rewards_per_episode.append(episode_reward)
    
    if (episode + 1) % 10 == 0:
        print(f"Episode {episode + 1}/{episodes}, Average Reward: {np.mean(rewards_per_episode[-10:])}")

## Results Visualization

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(rewards_per_episode)
plt.xlabel('Episode')
plt.ylabel('Reward')
plt.title('Q-Learning Training Progress')
plt.grid(True)
plt.show()